# safe-order-parser-ptbr: avaliação e fine-tuning

Roda quatro sistemas nos conjuntos de teste congelados e gera a tabela de resultados:
regras (já calculado), Qwen sem ajuste, Qwen com exemplos no prompt e Qwen com LoRA.

**Antes de rodar:** Ambiente de execução → Alterar tipo → **GPU T4**. Depois: Ambiente de execução → **Executar tudo**.
A primeira célula pede o zip do projeto. No fim, o notebook baixa `resultados.zip`.
Tempo total estimado: 35 a 60 minutos.

In [ ]:
# 1. Projeto: envie o zip do repositório (o mesmo que você descompacta no Codespaces)
import os, sys, zipfile, glob, subprocess, json, time, gc, random
from google.colab import files
if not os.path.exists("/content/proj/src"):
    up = files.upload()
    nome = next(iter(up))
    zipfile.ZipFile(nome).extractall("/content/proj")
os.chdir("/content/proj")
sys.path.insert(0, "/content/proj/src")
os.environ["PYTHONPATH"] = "/content/proj/src"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pydantic"], check=True)
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True).stdout[-200:])

In [ ]:
# 2. Modelo e funções de geração
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from order_parser.output_format import SISTEMA, mensagens, para_json
from order_parser.schema import Pedido

assert torch.cuda.is_available(), "Ligue a GPU: Ambiente de execução > Alterar tipo > T4"
MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 7
random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODELO)
tok.padding_side = "left"   # na geração em lote, o preenchimento fica à esquerda

def carregar_base(dtype):
    try:
        return AutoModelForCausalLM.from_pretrained(MODELO, dtype=dtype, device_map="cuda")
    except TypeError:   # versões antigas do transformers
        return AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=dtype, device_map="cuda")

CONJUNTOS = sorted(glob.glob("data/test/test_*.jsonl"))
TESTE = [json.loads(l) for f in CONJUNTOS for l in open(f, encoding="utf8")]
print(len(TESTE), "frases de teste em", len(CONJUNTOS), "conjuntos")
LAT = {}

@torch.no_grad()
def gerar(model, lista_msgs, max_new_tokens, lote=16):
    prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in lista_msgs]
    ordem = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    saidas = [None] * len(prompts); n_in = n_out = 0
    t0 = time.time()
    for k in range(0, len(ordem), lote):
        idx = ordem[k:k + lote]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        novos = out[:, enc["input_ids"].shape[1]:]
        for i, seq in zip(idx, novos):
            saidas[i] = tok.decode(seq, skip_special_tokens=True).strip()
            n_out += int((seq != tok.pad_token_id).sum())
        n_in += int(enc["attention_mask"].sum())
    dt = time.time() - t0
    return saidas, {"tokens_entrada": n_in / len(prompts), "tokens_saida": n_out / len(prompts), "seg_por_msg": dt / len(prompts)}

def avaliar(slug, nome, saidas, stats):
    os.makedirs("results", exist_ok=True)
    with open(f"results/preds_{slug}.jsonl", "w", encoding="utf8") as f:
        for ex, s in zip(TESTE, saidas):
            f.write(json.dumps({"id": ex["id"], "saida": s}, ensure_ascii=False) + "\n")
    LAT[slug] = stats
    json.dump(LAT, open("results/latency.json", "w"), indent=1)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", "--preds", f"results/preds_{slug}.jsonl", "--nome", nome, "--slug", slug],
                       capture_output=True, text=True)
    print(r.stdout[:1600], r.stderr[-800:])

In [ ]:
# 3. Qwen SEM ajuste (zero-shot): mesma instrução do sistema, nenhum exemplo
model = carregar_base(torch.float16)
model.eval()
saidas, stats = gerar(model, [mensagens(ex["texto"]) for ex in TESTE], max_new_tokens=256)
print("exemplo:", TESTE[0]["texto"], "->", saidas[0][:200])
avaliar("zero_shot", "Qwen sem ajuste", saidas, stats)

In [ ]:
# 4. Qwen com 8 exemplos no prompt (few-shot): a alternativa barata ao fine-tuning
TREINO = [json.loads(l) for l in open("data/train/train.jsonl", encoding="utf8")]
VAL = [json.loads(l) for l in open("data/train/val.jsonl", encoding="utf8")]

def escolher_exemplos(dados):
    r = lambda ex: ex["rotulo"]
    criterios = [
        lambda ex: r(ex)["intent"] == "emergencia",
        lambda ex: r(ex)["intent"] == "ataque",
        lambda ex: r(ex)["intent"] == "fora_escopo",
        lambda ex: r(ex)["intent"] == "pedido" and ex["saida"] == "{}",
        lambda ex: r(ex)["health_alert"] and len(r(ex)["restrictions"]) >= 1 and r(ex)["categories"],
        lambda ex: r(ex)["partial_restrictions"] and r(ex)["people"],
        lambda ex: r(ex)["exclude_categories"] and not r(ex)["categories"],
        lambda ex: r(ex)["max_price"] and r(ex)["free_delivery"] and r(ex)["categories"],
    ]
    return [next(ex for ex in dados if c(ex)) for c in criterios]

EXEMPLOS = escolher_exemplos(TREINO)
for ex in EXEMPLOS: print(ex["texto"], "->", ex["saida"])

def msgs_few_shot(texto):
    m = [{"role": "system", "content": SISTEMA}]
    for ex in EXEMPLOS:
        m += [{"role": "user", "content": ex["texto"]}, {"role": "assistant", "content": ex["saida"]}]
    return m + [{"role": "user", "content": texto}]

saidas, stats = gerar(model, [msgs_few_shot(ex["texto"]) for ex in TESTE], max_new_tokens=256, lote=8)
avaliar("few_shot", "Qwen + 8 exemplos no prompt", saidas, stats)
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Fine-tuning com LoRA
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

EPOCAS, LR, RANK = 2, 2e-4, 16
base = carregar_base(torch.float32)   # o treino usa precisão mista (fp16) por cima de pesos em 32 bits
cfg = LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, task_type="CAUSAL_LM",
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(base, cfg)
model.print_trainable_parameters()

class Dados(torch.utils.data.Dataset):
    # Cada exemplo vira: [instrução + mensagem] + [resposta certa]. A perda só conta na resposta:
    # os tokens da instrução recebem rótulo -100, que o PyTorch ignora.
    def __init__(self, exemplos):
        self.itens = []
        for ex in exemplos:
            prompt = tok.apply_chat_template(mensagens(ex["texto"]), tokenize=False, add_generation_prompt=True)
            p = tok(prompt, add_special_tokens=False)["input_ids"]
            r = tok(ex["saida"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
            self.itens.append({"input_ids": p + r, "labels": [-100] * len(p) + r})
    def __len__(self): return len(self.itens)
    def __getitem__(self, i): return self.itens[i]

def juntar(lote):
    n = max(len(x["input_ids"]) for x in lote)
    ids = torch.full((len(lote), n), tok.pad_token_id); lab = torch.full((len(lote), n), -100); att = torch.zeros((len(lote), n), dtype=torch.long)
    for i, x in enumerate(lote):   # no treino, o preenchimento fica à direita
        k = len(x["input_ids"])
        ids[i, :k] = torch.tensor(x["input_ids"]); lab[i, :k] = torch.tensor(x["labels"]); att[i, :k] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

d_treino, d_val = Dados(TREINO), Dados(VAL)
print("treino:", len(d_treino), "| validação:", len(d_val), "| tokens por exemplo:", sum(len(x["input_ids"]) for x in d_treino.itens) // len(d_treino))

kw = dict(output_dir="/content/saida_treino", num_train_epochs=EPOCAS, per_device_train_batch_size=8, per_device_eval_batch_size=16,
          gradient_accumulation_steps=2, learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=20,
          save_strategy="no", fp16=True, report_to="none", seed=SEED)
try:
    args = TrainingArguments(eval_strategy="epoch", **kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", **kw)
trainer = Trainer(model=model, args=args, train_dataset=d_treino, eval_dataset=d_val, data_collator=juntar)
t0 = time.time(); trainer.train(); minutos = (time.time() - t0) / 60
historico = trainer.state.log_history
json.dump({"modelo": MODELO, "epocas": EPOCAS, "lr": LR, "rank": RANK, "minutos_treino": minutos, "seed": SEED, "log": historico},
          open("results/training_log.json", "w"), indent=1)
print(f"treino: {minutos:.1f} min | perda de validação por época:", [round(h["eval_loss"], 4) for h in historico if "eval_loss" in h])

In [ ]:
# 6. Avaliação do modelo ajustado
model.eval()
saidas, stats = gerar(model, [mensagens(ex["texto"]) for ex in TESTE], max_new_tokens=128)
print("exemplo:", TESTE[0]["texto"], "->", saidas[0])
avaliar("lora", "Qwen + LoRA", saidas, stats)
model.save_pretrained("/content/proj/adapter")
tok.save_pretrained("/content/proj/adapter")

In [ ]:
# 7. Tabela final e download
print(subprocess.run([sys.executable, "scripts/compare.py"], capture_output=True, text=True).stdout)
with zipfile.ZipFile("/content/resultados.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pasta in ["results", "adapter"]:
        for raiz, _, arqs in os.walk(pasta):
            for a in arqs: z.write(os.path.join(raiz, a))
files.download("/content/resultados.zip")

## Opcional: publicar o adaptador no Hugging Face

1. Em huggingface.co → Settings → Access Tokens, crie um token do tipo **Write**.
2. No Colab, clique no ícone de chave (Secrets), crie `HF_TOKEN` com o token e ative o acesso do notebook.
3. Troque `SEU_USUARIO` abaixo e rode a célula. **Nunca cole o token no código.**

In [ ]:
from google.colab import userdata
REPO = "SEU_USUARIO/safe-order-parser-ptbr-qwen2.5-1.5b-lora"
if "SEU_USUARIO" not in REPO:
    model.push_to_hub(REPO, token=userdata.get("HF_TOKEN"), private=True)
    tok.push_to_hub(REPO, token=userdata.get("HF_TOKEN"), private=True)
    print("publicado (privado):", REPO)
else:
    print("Troque SEU_USUARIO pelo seu usuário do Hugging Face para publicar.")